In [1]:
import sys
from pathlib import Path

scripts_path = str(Path.cwd().parent / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from dart_client import analyze_company
print("✅ analyze_company import 성공!")
print(f"   함수 위치: {analyze_company.__module__}")

✅ analyze_company import 성공!
   함수 위치: dart_client.analyzer


In [2]:
import pandas as pd
from dart_client import analyze_company

test_companies = [
    {"name": "삼성전자",    "corp_code": "00126380", "stock_code": "005930"},
    {"name": "SK하이닉스", "corp_code": "00164779", "stock_code": "000660"},
    {"name": "카카오",      "corp_code": "00258801", "stock_code": "035720"},
    {"name": "셀트리온",    "corp_code": "00413046", "stock_code": "068270"},
    {"name": "KB금융",      "corp_code": "00688996", "stock_code": "105560"},
]

print("=" * 70)
print("📊 회귀 테스트 - 패키지화 후 5종목 (이전 결과와 같아야 함)")
print("=" * 70)

results = []
for company in test_companies:
    result = analyze_company(**company, year=2025)
    results.append(result)

# 결과 정리
success = [r for r in results if "error" not in r]
failed = [r for r in results if "error" in r]

print("\n" + "=" * 70)
print(f"성공: {len(success)} / 실패: {len(failed)}")
print("=" * 70)

if success:
    df = pd.DataFrame(success)
    cols = ["회사명", "주가", "EPS", "PER", "PBR", "ROE(%)", "시가총액(조)"]
    print(df[cols].to_string(index=False))

if failed:
    print("\n=== ❌ 실패 ===")
    for r in failed:
        print(f"  - {r['회사명']}: {r['error']}")

📊 회귀 테스트 - 패키지화 후 5종목 (이전 결과와 같아야 함)

[삼성전자] 분석 중... ✅ (주가 322,000원)

[SK하이닉스] 분석 중... ✅ (주가 2,215,000원)

[카카오] 분석 중... ✅ (주가 39,500원)

[셀트리온] 분석 중... ✅ (주가 170,000원)

[KB금융] 분석 중... ❌ 지표: 필수 데이터 누락: ['매출액']

성공: 4 / 실패: 1
   회사명      주가      EPS   PER   PBR  ROE(%)  시가총액(조)
  삼성전자  322000  7757.08 41.51  4.30   10.36   1876.6
SK하이닉스 2215000 61206.24 36.19 12.88   35.59   1554.2
   카카오   39500  1177.04 33.56  1.14    3.40     17.4
  셀트리온  170000  4718.31 36.03  2.14    5.94     37.2

=== ❌ 실패 ===
  - KB금융: 필수 데이터 누락: ['매출액']


In [1]:
# 노트북 Restart 후 (커널 메뉴 → Restart)

import sys
from pathlib import Path

scripts_path = str(Path.cwd().parent / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from dart_client import (
    search_company,
    get_company_by_stock_code,
    analyze_by_stock_code,
    analyze_by_search,
)

# 1. 검색 기능 테스트
print("=" * 60)
print("🔍 종목 검색 테스트")
print("=" * 60)

print("\n[1] '삼성' 검색:")
results = search_company("삼성", limit=5)
for r in results:
    print(f"  {r['stock_code']} {r['name']}")

print("\n[2] '카카오' 검색:")
results = search_company("카카오", limit=5)
for r in results:
    print(f"  {r['stock_code']} {r['name']}")

print("\n[3] 종목코드 '005930' 정확 조회:")
result = get_company_by_stock_code("005930")
print(f"  {result}")

print("\n[4] 종목코드로 분석 (한 줄!)")
result = analyze_by_stock_code("005930", year=2025)
print(f"  {result['회사명']}: EPS {result['EPS']}, PER {result['PER']}")

print("\n[5] 검색어로 분석:")
result = analyze_by_search("SK하이닉스", year=2025)
print(f"  {result['회사명']}: EPS {result['EPS']}, PER {result['PER']}")

🔍 종목 검색 테스트

[1] '삼성' 검색:
  052560 삼성수산
  108070 삼성디지털이미징
  000830 삼성물산
  009150 삼성전기
  068290 삼성출판사

[2] '카카오' 검색:
  016170 카카오엠
  293490 카카오게임즈
  035720 카카오
  377300 카카오페이
  323410 카카오뱅크

[3] 종목코드 '005930' 정확 조회:
  {'name': '삼성전자', 'stock_code': '005930', 'corp_code': '00126380'}

[4] 종목코드로 분석 (한 줄!)

[삼성전자] 분석 중... ✅ (주가 322,000원)
  삼성전자: EPS 7757.08, PER 41.51

[5] 검색어로 분석:

[SK하이닉스] 분석 중... ✅ (주가 2,215,000원)
  SK하이닉스: EPS 61206.24, PER 36.19


In [2]:
import sys
from pathlib import Path

scripts_path = str(Path.cwd().parent / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from dart_client import get_financial_data

# 삼성전자 4개 분기 데이터
quarters = [
    (2025, "11013", "2025_1Q"),
    (2025, "11012", "2025_2Q"),
    (2025, "11014", "2025_3Q"),
    (2025, "11011", "2025_FY"),
]

print("=" * 70)
print("📊 삼성전자 분기별 매출액 (단독값 검증)")
print("=" * 70)

results = []
for year, reprt_code, label in quarters:
    data = get_financial_data("00126380", year, reprt_code)
    if "error" not in data:
        results.append({
            "분기": label,
            "매출액(조)": round(data["매출액"] / 1e12, 2),
            "영업이익(조)": round(data["영업이익"] / 1e12, 2),
            "당기순이익(조)": round(data["당기순이익"] / 1e12, 2),
        })

for r in results:
    print(f"  {r['분기']}: 매출 {r['매출액(조)']:>7.2f}조, 영업이익 {r['영업이익(조)']:>6.2f}조, 순이익 {r['당기순이익(조)']:>6.2f}조")

# 검증: 1Q + 2Q + 3Q vs FY-4Q
print("\n=== 검증 ===")
if len(results) == 4:
    sum_1to3 = results[0]["매출액(조)"] + results[1]["매출액(조)"] + results[2]["매출액(조)"]
    fy = results[3]["매출액(조)"]
    q4_calc = fy - sum_1to3
    
    print(f"1Q + 2Q + 3Q 합계: {sum_1to3:.2f}조")
    print(f"연간 (FY):         {fy:.2f}조")
    print(f"4Q 계산값 (FY-합): {q4_calc:.2f}조")
    print()
    print(f"  → 4Q는 빼기로 계산해야 함 (약 {q4_calc:.0f}조)")

📊 삼성전자 분기별 매출액 (단독값 검증)
  2025_1Q: 매출   79.14조, 영업이익   6.69조, 순이익   8.22조
  2025_2Q: 매출   74.57조, 영업이익   4.68조, 순이익   5.12조
  2025_3Q: 매출   86.06조, 영업이익  12.17조, 순이익  12.23조
  2025_FY: 매출  333.61조, 영업이익  43.60조, 순이익  45.21조

=== 검증 ===
1Q + 2Q + 3Q 합계: 239.77조
연간 (FY):         333.61조
4Q 계산값 (FY-합): 93.84조

  → 4Q는 빼기로 계산해야 함 (약 94조)


In [1]:
import sys
import pandas as pd
from pathlib import Path

scripts_path = str(Path.cwd().parent / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from dart_client import get_quarterly_trend

# 삼성전자 2025년 분기별 추이
print("=" * 70)
print("📊 삼성전자 2025년 분기별 추이")
print("=" * 70)

trend = get_quarterly_trend("00126380", 2025)

# DataFrame으로 표 출력
df = pd.DataFrame([
    {
        "분기": q["quarter"],
        "매출액(조)": round(q["매출액"] / 1e12, 2) if q.get("매출액") else None,
        "영업이익(조)": round(q["영업이익"] / 1e12, 2) if q.get("영업이익") else None,
        "순이익(조)": round(q["당기순이익"] / 1e12, 2) if q.get("당기순이익") else None,
        "출처": q.get("출처", "에러"),
    }
    for q in trend
])
print(df.to_string(index=False))

# 시각화 (간단한 텍스트 차트)
print("\n=== 매출액 추이 ===")
max_val = max((q["매출액"] for q in trend if q.get("매출액")), default=1)
for q in trend:
    if q.get("매출액"):
        bar_len = int(q["매출액"] / max_val * 40)
        bar = "█" * bar_len
        print(f"  {q['quarter']}: {bar} {q['매출액']/1e12:.1f}조")

📊 삼성전자 2025년 분기별 추이
     분기  매출액(조)  영업이익(조)  순이익(조)             출처
2025_1Q   79.14     6.69    8.22          분기보고서
2025_2Q   74.57     4.68    5.12          분기보고서
2025_3Q   86.06    12.17   12.23          분기보고서
2025_4Q   93.84    20.07   19.64 사업보고서 - 1~3분기합

=== 매출액 추이 ===
  2025_1Q: █████████████████████████████████ 79.1조
  2025_2Q: ███████████████████████████████ 74.6조
  2025_3Q: ████████████████████████████████████ 86.1조
  2025_4Q: ████████████████████████████████████████ 93.8조
